In [1]:
# %% [markdown]
# # Notebook 4: Leakage-Safe Feature Engineering

# %%
import os
import numpy as np
import pandas as pd

# %%
# 1. Load master dataset
candidates = [
    "../data/processed/delhi_pollution_weather_daily_2020_2025.csv",
    "data/processed/delhi_pollution_weather_daily_2020_2025.csv",
    "delhi_pollution_weather_daily_2020_2025.csv"
]
df = pd.read_csv(next(p for p in candidates if os.path.exists(p)))
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)
data = df.copy()

# %%
# 2. Target Creation: Forecast Next-Day PM2.5 (t+1)
data["target_next_day"] = data["PM2.5"].shift(-1)
data["target_date"] = data["Date"].shift(-1)

# %%
# 3. Autoregressive Lag Features (Information available up to day t)
for lag in [1, 2, 3, 7]:
    data[f"pm25_lag_{lag}"] = data["PM2.5"].shift(lag)

# 4. Rolling Statistics (Shifted by 1 day to strictly prevent leakage)
# pm25_roll3 contains the average over days [t-3, t-2, t-1]; day t PM2.5 is preserved in the base column
data["pm25_roll3"] = data["PM2.5"].shift(1).rolling(3, min_periods=2).mean()
data["pm25_roll7"] = data["PM2.5"].shift(1).rolling(7, min_periods=4).mean()

# %%
# 5. Meteorological Transformations & Atmospheric Dynamics
data["pressure_change"] = data["pressure_mean"].diff(1)
data["temp_range"] = data["temperature_max"] - data["temperature_min"]

# Circular month representation
data["month_sin"] = np.sin(2 * np.pi * data["month"] / 12)
data["month_cos"] = np.cos(2 * np.pi * data["month"] / 12)

# %%
# 6. Audit and Drop the Final Row (which has no target for next day)
data = data.dropna(subset=["target_next_day", "target_date"]).reset_index(drop=True)

print(f"Engineered dataset shape: {data.shape}")
print(f"Target date bounds: {data['target_date'].min().strftime('%Y-%m-%d')} to {data['target_date'].max().strftime('%Y-%m-%d')}")

# %%
# 7. Export engineered features
out_dirs = ["../data/processed", "data/processed", "."]
target_dir = next((d for d in out_dirs if os.path.exists(d)), "../data/processed")
os.makedirs(target_dir, exist_ok=True)
out_file = os.path.join(target_dir, "delhi_engineered_features.csv")

data.to_csv(out_file, index=False)
print(f"Features saved successfully to: {out_file}")

Engineered dataset shape: (2129, 38)
Target date bounds: 2020-01-02 to 2025-12-31
Features saved successfully to: ../data/processed\delhi_engineered_features.csv
